![Universidad Iberoamericana del Ecuador](https://raw.githubusercontent.com/andresrubiop/ml-unibe-estudiantes/main/utils/logo-unibe-color.png)

<div style="background:#004890;color:#FFFFFF;padding:14px 18px;border-radius:10px;margin:6px 0 12px 0;border-bottom:4px solid #FCC000"><div style="font-size:12px;letter-spacing:.08em;text-transform:uppercase;opacity:.9">Machine Learning · Trabajo final · Producto final · 30 % de la nota</div><div style="font-size:22px;font-weight:700;margin-top:4px">Trabajo final · Proyecto integrador de machine learning sobre un caso de Big Data</div><div style="font-size:12px;opacity:.9;margin-top:4px">Posgrado · Maestría en Big Data y Ciencia de Datos · UNIB.E</div></div>

| | |
|---|---|
| **Qué entregas** | **Solo este notebook ejecutado** (`.ipynb`), renombrado `TF_Apellido_Nombre.ipynb`, en el aula virtual. **El notebook es tu informe**: no hay documento ni presentación aparte. |
| **Fecha** | **Jueves 19 de noviembre de 2026** |
| **Trabajo** | Individual |
| **Peso** | 3 puntos (30 % de la nota final) · rúbrica al final de este notebook y en el aula |
| **Uso de IA** | **Permitido sin límite**: Claude Code, Codex, ChatGPT, Gemini en Colab, Copilot u otra herramienta, siempre que lo **declares** en la sección 12. |
| **Teoría** | Todo el módulo: M1 (supervisado y evaluación) y M2 (no supervisado, conocimiento y escala) |

**Qué vas a hacer.** Resolver de principio a fin un caso de Big Data: plantear el problema, preparar los datos sin fugas, comparar al menos tres modelos con validación cruzada, aplicar una técnica no supervisada o de extracción del conocimiento, evaluar una sola vez en la prueba y explicar el resultado a quien decide.

**Cómo está armada esta plantilla.** Ya trae un **caso de ejemplo que funciona** (ingresos de adultos del censo de EE. UU., 48 842 personas, de OpenML) para que veas el flujo completo. **Cámbialo por tu caso** en la sección 2: el resto de las celdas está escrito para adaptarse a cualquier tabla con una columna objetivo. Si aún no tienes caso, la lista de la sección 2 trae conjuntos públicos.

**Secciones:** 0 · cómo trabajar → 1 · ficha del proyecto → 2 · datos → 3 · exploración → 4 · partición sin fugas y baseline → 5 · preparación → 6 · tres modelos con validación cruzada → 7 · técnica no supervisada o de conocimiento → 8 · evaluación final → 9 · interpretación → 10 · conclusiones para quien decide → 11 · reproducibilidad → 12 · declaración de IA → lista de control.

## 0 · Cómo trabajar este notebook

1. **Guarda tu copia.** En Colab: *Archivo → Guardar una copia en Drive*. Trabaja siempre sobre esa copia.
2. **Ejecuta las secciones 1 a 5 con el ejemplo** para ver cómo funciona (tarda un minuto). Desde la sección 6 hay celdas `# TODO` por completar: la ejecución se detiene en la primera autoverificación que no pase, y eso es normal.
3. **Cambia los datos** en la sección 2 por los de tu caso y ajusta `OBJETIVO` y `TAREA`. Vuelve a ejecutar y completa las celdas `# TODO`.
4. **Escribe en las celdas ✍️.** Son tu informe: escríbelas para alguien que decide (un gerente, un médico, un analista de negocio), no para un programador. Usa las cifras de tus celdas.
5. **Antes de entregar:** *Reiniciar y ejecutar todas*. Si todo corre sin errores y sin el texto de ejemplo, descarga con *Archivo → Descargar → Descargar .ipynb*, renómbralo `TF_Apellido_Nombre.ipynb` y súbelo al aula virtual.

> **¿Datos de tu trabajo?** Solo con permiso de la organización y **anonimizados**: sin nombres, cédulas, correos, teléfonos ni direcciones. Si no puedes compartirlos, usa un conjunto público parecido.

## 1 · Ficha del proyecto

✍️ Completa la tabla (doble clic para editar):

| | |
|---|---|
| **Título** | *por ejemplo: ¿Quién gana más de 50 000 dólares al año?* |
| **Pregunta de negocio** | *qué decisión se quiere mejorar y quién la toma* |
| **Tipo de tarea** | *clasificación binaria, multiclase o regresión; y la técnica no supervisada que agregarás* |
| **Qué es una fila** | *una persona, una transacción, un cliente por mes…* |
| **Variable objetivo** | *qué se predice y en qué unidad* |
| **Fuente de los datos** | *enlace o institución; fecha de descarga* |
| **Licencia o permiso** | *licencia pública o permiso de la organización; cómo se anonimizó* |
| **Volumen** | *filas, columnas y tamaño en disco* |
| **Costo de cada error** | *qué pasa si el modelo se equivoca en un sentido o en el otro* |

## 2 · Datos

### Configuración

In [ ]:
import os
import platform
import time
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import sklearn
from cycler import cycler

from sklearn.exceptions import ConvergenceWarning

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=ConvergenceWarning)   # avisos de convergencia de la logística: no cambian las conclusiones
SEED = 2026
UNIBE = {"azul": "#004890", "naranja": "#C2410C", "dorado": "#A87900", "verde": "#3A7D44", "gris": "#77787B"}
plt.rcParams.update({
    "axes.prop_cycle": cycler(color=list(UNIBE.values())), "axes.titlecolor": UNIBE["azul"],
    "axes.titleweight": "bold", "axes.edgecolor": UNIBE["gris"], "axes.grid": True, "grid.color": "#E8EDF2",
    "axes.spines.top": False, "axes.spines.right": False, "figure.dpi": 110, "legend.frameon": False,
})
print(f"Python {platform.python_version()} · scikit-learn {sklearn.__version__} · pandas {pd.__version__} · semilla {SEED}")

### Cómo cargar tus datos

Elige **una** opción y reemplaza la celda de carga de abajo. Todas terminan en un `DataFrame` llamado `df`.

**A · Un archivo en internet (CSV o Parquet)**
```python
df = pd.read_csv("https://.../datos.csv")              # separador distinto: sep=";" · decimales con coma: decimal=","
df = pd.read_parquet("https://.../datos.parquet")
```

**B · Un archivo de tu computadora (se sube a Colab)**
```python
from google.colab import files
subido = files.upload()                                 # abre un selector de archivos
df = pd.read_csv(next(iter(subido)))                    # o pd.read_excel(...) para .xlsx
```
Lo subido se borra al cerrar la sesión de Colab: para archivos grandes o que usarás varias veces, usa Drive.

**C · Un archivo en tu Google Drive** (recomendado para archivos grandes)
```python
from google.colab import drive
drive.mount("/content/drive")                           # pide permiso la primera vez
df = pd.read_csv("/content/drive/MyDrive/carpeta/datos.csv")
```

**D · Un conjunto de OpenML** (miles de conjuntos públicos, por nombre o número)
```python
from sklearn.datasets import fetch_openml
df = fetch_openml("credit-g", version=1, as_frame=True).frame   # busca en https://www.openml.org/search?type=data
```

**E · Kaggle:** descarga el archivo desde la página del conjunto (botón *Download*) y súbelo con la opción B o C. No pegues tu clave de Kaggle en el notebook que vas a entregar.

**F · Datos muy grandes (varios GB)**
```python
df = pd.read_csv("datos.csv", usecols=["col1", "col2", "objetivo"], dtype={"col1": "float32"})  # solo las columnas necesarias
df = pd.read_csv("datos.csv", nrows=200_000)                                               # o una muestra para empezar
df = pd.read_parquet("carpeta_parquet/", columns=["col1", "objetivo"])                     # Parquet lee solo lo que pides
```
Si de verdad no cabe en memoria, usa Spark como en el Deber 3 (ruta A) y documenta por qué.

<details><summary><b>Conjuntos públicos sugeridos si aún no tienes caso</b></summary>

| Conjunto | Tarea | Filas | Cómo cargarlo |
|---|---|---|---|
| Ingresos de adultos (censo de EE. UU.) | clasificación | 48 842 | `fetch_openml("adult", version=2, as_frame=True)` |
| Crédito alemán | clasificación (riesgo) | 1 000 | `fetch_openml("credit-g", version=1, as_frame=True)` |
| Marketing bancario | clasificación | 45 211 | `fetch_openml("bank-marketing", version=1, as_frame=True)` |
| Precios de viviendas en California | regresión | 20 640 | `from sklearn.datasets import fetch_california_housing` |
| Demanda de bicicletas compartidas | regresión | 17 379 | `fetch_openml("Bike_Sharing_Demand", version=2, as_frame=True)` |
| Cobertura forestal | multiclase | 581 012 | `from sklearn.datasets import fetch_covtype` |
</details>

In [ ]:
from sklearn.datasets import fetch_openml

# ===== Cambia esta celda por tus datos =====
datos = fetch_openml("adult", version=2, as_frame=True)       # ejemplo: ingresos de adultos (OpenML 1590)
df = datos.frame
OBJETIVO = "class"                  # nombre de la columna que se predice
TAREA = "clasificacion"             # "clasificacion" o "regresion"
# ============================================
print(f"{df.shape[0]:,} filas × {df.shape[1]} columnas · {df.memory_usage(deep=True).sum() / 1e6:.1f} MB en memoria")
df.head()

**Muestra para trabajar rápido (opcional).** Si tu conjunto tiene más de 20 000 filas, trabaja primero con una muestra fija y, al final, si el tiempo lo permite, vuelve a ejecutar con todo. Deja escrito qué usaste.

In [ ]:
MUESTRA = 15_000                    # None = usar todas las filas
if MUESTRA and len(df) > MUESTRA:
    df = df.sample(MUESTRA, random_state=SEED).reset_index(drop=True)
print(f"trabajando con {len(df):,} filas")

**✅ Autoverificación de los datos**

In [ ]:
assert OBJETIVO in df.columns, f"La columna objetivo «{OBJETIVO}» no está en df."
assert TAREA in ("clasificacion", "regresion"), "TAREA debe ser 'clasificacion' o 'regresion'."
assert df[OBJETIVO].notna().all(), "Hay filas sin objetivo: quítalas con df = df.dropna(subset=[OBJETIVO])."
print("✓ Datos listos")

## 3 · Exploración

Antes de modelar, conoce los datos: tipos de columnas, valores faltantes, distribución del objetivo y relaciones evidentes. La celda siguiente hace el resumen básico.

In [ ]:
resumen = pd.DataFrame({"tipo": df.dtypes.astype(str), "faltantes (%)": (df.isna().mean() * 100).round(1),
                        "valores distintos": df.nunique()})
print(resumen.to_string())
if TAREA == "clasificacion":
    print("\nDistribución del objetivo:\n", df[OBJETIVO].value_counts(normalize=True).round(3).to_string())
else:
    print("\nObjetivo:\n", df[OBJETIVO].describe().round(2).to_string())

In [ ]:
# TODO: dos gráficos que te ayuden a entender el problema (por ejemplo, el objetivo según una variable importante).
fig, axs = plt.subplots(1, 2, figsize=(11, 3.4))
...
plt.show()

✍️ **Lo que muestran los datos.** Tres o cuatro observaciones: ¿hay clases desbalanceadas? ¿columnas con muchos faltantes? ¿alguna variable que **no existiría al momento de predecir** (fuga) y que hay que quitar?

*Tu texto aquí.*

## 4 · Partición sin fugas y baseline

La **prueba se separa ahora** y no se vuelve a tocar hasta la sección 8. Elige el tipo de partición según tus datos:

- **Aleatoria estratificada** (la de abajo): filas independientes.
- **Por grupos** (`GroupShuffleSplit`): si hay varias filas del mismo cliente, paciente o máquina, que no queden a ambos lados.
- **Temporal**: si los datos tienen fecha, entrena con el pasado y prueba con lo más reciente.

También quita aquí las columnas que serían **fuga** o que no deben usarse (identificadores, datos personales).

In [ ]:
from sklearn.dummy import DummyClassifier, DummyRegressor
from sklearn.model_selection import train_test_split

COLUMNAS_A_QUITAR = ["fnlwgt"]      # TODO: identificadores, fugas o datos personales de tu caso
X = df.drop(columns=[OBJETIVO] + [c for c in COLUMNAS_A_QUITAR if c in df.columns])
y = df[OBJETIVO]
if TAREA == "clasificacion":
    y = y.astype(str)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=SEED, stratify=y if TAREA == "clasificacion" else None)
print(f"entrenamiento: {len(X_train):,} · prueba: {len(X_test):,} · {X.shape[1]} variables")

if TAREA == "clasificacion":
    METRICA = "balanced_accuracy"   # TODO: elige la métrica según el costo del error (f1, recall, roc_auc, balanced_accuracy…)
    baseline = DummyClassifier(strategy="most_frequent").fit(X_train, y_train)
else:
    METRICA = "neg_mean_absolute_error"   # o "r2", "neg_root_mean_squared_error"
    baseline = DummyRegressor(strategy="mean").fit(X_train, y_train)
from sklearn.metrics import get_scorer

puntaje_baseline = get_scorer(METRICA)(baseline, X_train, y_train)
print(f"métrica: {METRICA} · baseline en entrenamiento: {puntaje_baseline:.3f}")

✍️ **¿Por qué esta métrica?** Relaciónala con el costo de cada error de la ficha del proyecto. ¿Qué significa la cifra del baseline?

*Tu texto aquí.*

## 5 · Preparación dentro de un pipeline

La preparación (imputar faltantes, escalar, codificar categorías) va **dentro** de un `Pipeline`: así la validación cruzada la aprende solo con los datos de entrenamiento de cada pliegue y no hay fugas. La celda detecta sola las columnas numéricas y categóricas.

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

numericas = X_train.select_dtypes(include="number").columns.tolist()
categoricas = [c for c in X_train.columns if c not in numericas]
preparacion = ColumnTransformer([
    ("num", make_pipeline(SimpleImputer(strategy="median"), StandardScaler()), numericas),
    ("cat", make_pipeline(SimpleImputer(strategy="most_frequent"),
                          OneHotEncoder(handle_unknown="ignore", min_frequency=0.01, sparse_output=False)), categoricas),
])
print(f"{len(numericas)} numéricas: {numericas}\n{len(categoricas)} categóricas: {categoricas}")

## 6 · Tres modelos con validación cruzada

Compara **al menos tres modelos de las dos sesiones**, elegidos con argumentos (uno simple e interpretable, uno no lineal y uno de conjunto es una buena combinación). Si ajustas hiperparámetros, usa **validación cruzada anidada**: un `GridSearchCV` (interno) dentro de `cross_val_score` (externo). La cifra externa es la estimación honesta.

Te dejamos el primer modelo hecho: agrega los otros dos.

In [ ]:
from sklearn.ensemble import HistGradientBoostingClassifier, HistGradientBoostingRegressor, RandomForestClassifier, RandomForestRegressor
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.model_selection import GridSearchCV, KFold, StratifiedKFold, cross_val_score
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor

Particion = StratifiedKFold if TAREA == "clasificacion" else KFold
cv_interna = Particion(n_splits=3, shuffle=True, random_state=SEED)
cv_externa = Particion(n_splits=3, shuffle=True, random_state=SEED + 1)
clasif = TAREA == "clasificacion"

modelos = {
    "Lineal (logística o Ridge)": (
        Pipeline([("prep", preparacion), ("modelo", LogisticRegression(max_iter=2000) if clasif else Ridge())]),
        {"modelo__C": [0.1, 1.0]} if clasif else {"modelo__alpha": [1.0, 10.0]},
    ),
}
# TODO: agrega dos modelos más (por ejemplo, un árbol de decisión y un bosque aleatorio o gradient boosting), cada uno con su rejilla.
modelos["Árbol de decisión"] = (
    Pipeline([("prep", preparacion), ("modelo", DecisionTreeClassifier(random_state=SEED))]),
    {"modelo__max_depth": [...]},
)
modelos["..."] = (...)

filas = []
for nombre, (pipe, rejilla) in modelos.items():
    t0 = time.time()
    interna = GridSearchCV(pipe, rejilla, cv=cv_interna, scoring=METRICA, n_jobs=-1)
    externos = cross_val_score(interna, X_train, y_train, cv=cv_externa, scoring=METRICA, n_jobs=-1)
    filas.append({"modelo": nombre, f"{METRICA} (media)": externos.mean(), "± desviación": externos.std(), "tiempo (s)": time.time() - t0})
    print(f"{nombre:28s} {externos.mean():.3f} ± {externos.std():.3f}  ({time.time() - t0:.0f} s)")
tabla = pd.DataFrame(filas).sort_values(f"{METRICA} (media)", ascending=False).reset_index(drop=True)
tabla.round(3)

**✅ Autoverificación de la sección 6**

In [ ]:
assert len(tabla) >= 3, "Compara al menos tres modelos."
assert tabla[f"{METRICA} (media)"].max() > puntaje_baseline, "Ningún modelo supera al baseline: revisa la preparación o la métrica."
print("✓ Sección 6 correcta")

✍️ **Elección de modelos.** ¿Por qué elegiste estos tres? ¿Cuál gana y la diferencia es mayor que la desviación? ¿Vale la pena el más complejo frente al más simple?

*Tu texto aquí.*

## 7 · Técnica no supervisada o de extracción del conocimiento

Agrega **al menos una** de estas técnicas de la sesión 2 y explica qué aporta a tu caso:

- **Clustering** (k-means, mezcla de gaussianas o DBSCAN) para encontrar perfiles o segmentos;
- **Reducción de dimensionalidad** (PCA) para visualizar o simplificar;
- **Detección de anomalías** (Isolation Forest) para encontrar casos raros;
- **Selección de características** (filtro, envoltura o incrustada) para quedarte con las variables que importan;
- **Reglas** de un árbol corto (`export_text`) para explicar el problema con pocas condiciones.

El ejemplo de abajo busca **perfiles** con k-means sobre las variables numéricas del entrenamiento.

In [ ]:
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

# TODO: aplica tu técnica (cámbiala si otra tiene más sentido para tu caso).
...

✍️ **Qué aporta.** Describe los perfiles, reglas o variables que encontraste con palabras de negocio. ¿Cambia en algo la decisión?

*Tu texto aquí.*

## 8 · Evaluación final, una sola vez

Ajusta el **mejor modelo según la validación** con todo el entrenamiento (con su búsqueda de hiperparámetros) y evalúalo **una sola vez** en la prueba. Agrega la matriz de confusión (clasificación) o el gráfico de predicho frente a real (regresión).

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay, classification_report

nombre_mejor = tabla.loc[0, "modelo"]
pipe, rejilla = modelos[nombre_mejor]
final = GridSearchCV(pipe, rejilla, cv=cv_interna, scoring=METRICA, n_jobs=-1).fit(X_train, y_train)
puntaje_prueba = get_scorer(METRICA)(final, X_test, y_test)
print(f"{nombre_mejor} · hiperparámetros {final.best_params_} · {METRICA} en la prueba = {puntaje_prueba:.3f}")
if clasif:
    print(classification_report(y_test, final.predict(X_test), digits=3))
    ConfusionMatrixDisplay.from_estimator(final, X_test, y_test, cmap="Blues", colorbar=False); plt.show()
else:
    plt.scatter(y_test, final.predict(X_test), s=6, alpha=0.4); plt.xlabel("real"); plt.ylabel("predicho"); plt.show()

✍️ **¿La prueba confirma la validación?** Compara la cifra de la prueba con la de la sección 6. ¿Qué errores comete el modelo y cuál es más costoso?

*Tu texto aquí.*

## 9 · Interpretación

La **importancia por permutación** mide cuánto empeora el modelo si se desordena cada variable: sirve para cualquier modelo. La celda la calcula sobre la prueba.

In [ ]:
from sklearn.inspection import permutation_importance

imp = permutation_importance(final, X_test, y_test, scoring=METRICA, n_repeats=5, random_state=SEED, n_jobs=-1)
importancias = pd.Series(imp.importances_mean, index=X_test.columns).sort_values()
importancias.tail(10).plot.barh(color=UNIBE["azul"])
plt.title("Las 10 variables más importantes (permutación)"); plt.xlabel(f"caída de {METRICA}"); plt.show()

✍️ **Interpretación para quien decide.** ¿Qué variables pesan más y tiene sentido? ¿Hay alguna que preocupe por **sesgo** o **discriminación** (sexo, edad, origen)? ¿Qué harías al respecto?

*Tu texto aquí.*

## 10 · Conclusiones para quien decide

✍️ Escribe entre 250 y 400 palabras con esta estructura:

1. **Respuesta a la pregunta de negocio** en una o dos frases, con la cifra principal.
2. **Qué modelo recomiendas y por qué** (desempeño, tiempo, interpretabilidad).
3. **Qué aprendiste de los datos** (variables, perfiles o reglas de las secciones 7 y 9).
4. **Limitaciones y riesgos:** calidad de los datos, sesgos, cambios en el tiempo, uso indebido.
5. **Escala:** ¿cabe en scikit-learn o haría falta Spark, PyTorch o un clúster con todos los datos?
6. **Siguientes pasos.**

*Tu texto aquí.*

## 11 · Reproducibilidad

La celda registra las versiones con las que corrió tu notebook. Además, deja claro en el texto de la sección 2 de dónde salen los datos.

In [ ]:
import importlib.metadata as md

paquetes = ["scikit-learn", "pandas", "numpy", "matplotlib"]
print(f"Python {platform.python_version()} · semilla {SEED} · filas usadas {len(df):,}")
for p in paquetes:
    print(f"{p} {md.version(p)}")

## 12 · Declaración de uso de IA

En este módulo **se permite el uso total de la IA** (Claude Code, Codex, ChatGPT, Gemini, Copilot u otra) **siempre que lo declares**. Completa la tabla aunque no la hayas usado:

| | |
|---|---|
| **Herramientas y versión** | *por ejemplo: Claude Code y Gemini en Colab, noviembre de 2026* |
| **Para qué la usaste** | *código, búsqueda de datos, depuración, redacción, revisión…* |
| **Prompts principales** | *copia los tres o cuatro más importantes* |
| **Qué verificaste tú** | *cómo comprobaste que el código y las cifras eran correctos* |
| **Qué corregiste** | *errores de la IA que encontraste y cómo los arreglaste* |

## Lista de control antes de entregar

- [ ] Cambiaste el caso de ejemplo por el tuyo (o lo justificaste) y completaste la ficha del proyecto.
- [ ] *Reiniciar y ejecutar todas* corre sin errores y las autoverificaciones muestran ✓.
- [ ] La prueba se separó en la sección 4 y se usó **una sola vez**, en la sección 8.
- [ ] Todas las celdas ✍️ tienen tu texto (no quedó ningún «Tu texto aquí»).
- [ ] El archivo se llama `TF_Apellido_Nombre.ipynb` y lo subiste al aula virtual antes del jueves 19/11.

**Rúbrica (3 puntos, sílabo sección 8):**

| Criterio | Qué se espera | Puntos |
|---|---|---|
| Problema y preparación de datos | Problema bien formulado, datos descritos, preparación dentro de pipelines, sin fugas | 0.5 |
| Selección justificada de modelos | Al menos tres modelos adecuados, con argumentos, y una técnica no supervisada o de conocimiento | 0.5 |
| Validación cruzada y métricas | Esquema sin fugas, métrica coherente con el costo, una sola evaluación en la prueba | 0.8 |
| Interpretación y comunicación | Conocimiento útil para quien decide; textos claros en el notebook | 0.7 |
| Reproducibilidad e IA declarada | El notebook corre completo; versiones registradas; IA declarada y verificada | 0.5 |